# 05 · CTGAN: selección de hiperparámetros

Entrena CTGAN **solo con la clase minoritaria** del entrenamiento interno (escala original), genera M − m registros y elige (épocas, tamaño de lote) por el AUC-PR en validación del proxy de regresión logística.

In [ ]:
import json
import random
import joblib
import numpy as np
import pandas as pd
import torch
from collections import Counter
from ctgan import CTGAN
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)
from utils.config_loader import load_config

config = load_config("dataset_config.yaml")
RANDOM_STATE = config.get("random_state", 42)
print("Dataset:", config["dataset_name"])
USE_GPU = torch.cuda.is_available()
scaler = joblib.load("outputs/data/scaler.pkl")
categorical_cols = config.get("categorical_features", [])
PARAM_GRID = [
    {"epochs": 300, "batch_size": 500, "pac": 10},
    {"epochs": 500, "batch_size": 500, "pac": 10},
    {"epochs": 500, "batch_size": 1000, "pac": 10},
]


def set_seed(seed):
    """Fija las semillas de Python, NumPy y PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Datos

In [ ]:
SPLIT_DIR = "outputs/splits"

X_train_raw = pd.read_csv(f"{SPLIT_DIR}/X_train_raw_in.csv")      # entrenamiento interno, escala original
X_train_proc = pd.read_csv(f"{SPLIT_DIR}/X_train_in.csv")         # entrenamiento interno, escalado
y_train = pd.read_csv(f"{SPLIT_DIR}/y_train_in.csv")["y"].values
X_val_proc = pd.read_csv(f"{SPLIT_DIR}/X_val_in.csv")             # validación, escalada
y_val = pd.read_csv(f"{SPLIT_DIR}/y_val_in.csv")["y"].values

counter = Counter(y_train)
minority = min(counter, key=counter.get)
majority = max(counter, key=counter.get)
n_to_generate = counter[majority] - counter[minority]
print(f"Distribución del entrenamiento interno: { {int(k): int(v) for k, v in counter.items()} }")
print(f"Registros sintéticos a generar (M - m): {n_to_generate}")
X_minority_raw = X_train_raw[y_train == minority].copy()

## Búsqueda de hiperparámetros

In [ ]:
def proxy_auc_pr(synth_raw):
    """AUC-PR en validación de una regresión logística entrenada con datos reales + sintéticos."""
    synth_proc = synth_raw.copy()
    num_cols = [c for c in synth_raw.columns if c not in categorical_cols]
    synth_proc[num_cols] = scaler.transform(synth_raw[num_cols].astype(float))
    X_proxy = pd.concat([X_train_proc, synth_proc], ignore_index=True)
    y_proxy = np.concatenate([y_train, np.full(len(synth_proc), minority)])
    proxy = LogisticRegression(max_iter=3000)
    proxy.fit(X_proxy, y_proxy)
    return average_precision_score(y_val, proxy.predict_proba(X_val_proc)[:, 1])


best_score, best_params, best_synth = -np.inf, None, None
for params in PARAM_GRID:
    set_seed(RANDOM_STATE)
    model = CTGAN(epochs=params["epochs"], batch_size=params["batch_size"], pac=params["pac"],
                  verbose=False, enable_gpu=USE_GPU)
    model.fit(X_minority_raw, categorical_cols)
    synth_raw = model.sample(n_to_generate)

    auc_pr = proxy_auc_pr(synth_raw)
    print(f"{params}: AUC-PR validación = {auc_pr:.4f}")
    if auc_pr > best_score:
        best_score, best_params, best_synth = auc_pr, params, synth_raw.copy()

## Guardado

In [ ]:
OUTPUT_DIR = "outputs/ctgan"
os.makedirs(OUTPUT_DIR, exist_ok=True)
X_final = pd.concat([X_train_raw, best_synth], ignore_index=True)
y_final = np.concatenate([y_train, np.full(len(best_synth), minority)])
X_final.to_csv(f"{OUTPUT_DIR}/X_train_ctgan.csv", index=False)
pd.DataFrame({"y": y_final}).to_csv(f"{OUTPUT_DIR}/y_train_ctgan.csv", index=False)

with open(f"{OUTPUT_DIR}/best_config.json", "w") as f:
    json.dump({"best_auc_pr": float(best_score), "best_params": best_params,
               "training_mode": "minority_only"}, f, indent=4)
print(f"Configuración seleccionada: {best_params} (AUC-PR = {best_score:.4f})")